In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from model.actor_critic import FrameObservationEncoderNet, EncoderNet
from dataset import get_dataloader

from tqdm import trange
import tqdm

In [2]:
def cosine_loss(x, y):
    return 1. - F.cosine_similarity(x, y, dim=-1).mean()

def mse_loss(x, y):
    return F.mse_loss(x, y, reduction="mean")

In [3]:
class Alignment(nn.Module):
    def __init__(self, state_encoder, frame_encoder, state_feature_layer=-1):
        super().__init__()

        self.state_feature_layer = state_feature_layer

        self.state_encoder = state_encoder

        self.frame_encoder = frame_encoder

        self.state_encoder.eval()

        for param in self.state_encoder.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def encode_states(self, vectors):
        state_features = self.state_encoder.get_features(vectors)
        return state_features[self.state_feature_layer]

    def encode_frames(self, frames):
        frame_features = self.frame_encoder(frames, False, False)

        return frame_features
    
    def forward(self, frames, states):
        frame_features = self.encode_frames(frames)
        state_features = self.encode_states(states)

        return frame_features, state_features

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 20
encoder_weight, actor_weight, critic_wegit = torch.load("state_model.pth", weights_only=True)

state_encoder = EncoderNet(6+6+3+4+3+4, [256, 256]).to(device)
frame_encoder = FrameObservationEncoderNet(6, state_encoder.dim).to(device)

state_encoder.load_state_dict(encoder_weight)

model = Alignment(state_encoder, frame_encoder, -1).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=epochs)
dataloader = get_dataloader()
size = len(dataloader.dataset)
print(size)

Using cache found in /home/xdang/.cache/torch/hub/pytorch_vision_v0.10.0
/home/xdang/miniconda3/envs/isaaclab/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/xdang/miniconda3/envs/isaaclab/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


100000


In [5]:
for _ in trange(10, desc="Epochs"):
    running_loss = 0.0
    running_cosine_loss = 0.0
    for _, (vectors, frames) in enumerate(dataloader):
        vectors = vectors.to(device)
        frames = frames.to(device)
        
        frame_features, vector_features = model(frames, vectors)
        mse_losss_value = mse_loss(frame_features, vector_features)
        cosine_loss_value = cosine_loss(frame_features, vector_features)
        loss = 1 * mse_losss_value + 0. * cosine_loss_value
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * vectors.size(0)
        running_cosine_loss += cosine_loss_value.item() * vectors.size(0)
    
    scheduler.step()
    train_loss = running_loss / size
    train_cosine_loss = running_cosine_loss / size
    tqdm.tqdm.write(f"Train Loss: {train_loss:.4f}, Cosine Loss: {train_cosine_loss:.4f}")

torch.save([model.frame_encoder.state_dict(), actor_weight, critic_wegit], "frame_model.pth")

Epochs:  10%|█         | 1/10 [02:57<26:41, 177.91s/it]

Train Loss: 0.2769, Cosine Loss: 0.0987


Epochs:  20%|██        | 2/10 [05:59<24:01, 180.16s/it]

Train Loss: 0.0972, Cosine Loss: 0.0326


Epochs:  30%|███       | 3/10 [09:02<21:08, 181.19s/it]

...

Train Loss: 0.0489, Cosine Loss: 0.0168


Epochs:  90%|█████████ | 9/10 [27:09<03:01, 181.16s/it]

Train Loss: 0.0465, Cosine Loss: 0.0159


Epochs: 100%|██████████| 10/10 [30:06<00:00, 180.62s/it]

Train Loss: 0.0434, Cosine Loss: 0.0148


In [6]:
for _, (vectors, frames) in enumerate(dataloader):
    vectors = vectors.to(device)
    frames = frames.to(device)
    frame_features, vector_features = model(frames, vectors)
    cosine_loss_value = cosine_loss(frame_features, vector_features)

    print(cosine_loss_value)
   

tensor(0.0584, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0594, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0626, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0604, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0548, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0560, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0581, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0578, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0629, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0563, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0597, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0523, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0561, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0573, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0546, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0549, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0524, device='cuda:0', grad_fn=<RsubBackward1>)
tensor(0.0578, device='cuda:0',